# Task 2 — Notebook 01: CDL time series (processed Parquet)

**NAFSI Task 2:** ten-year CDL sequences (here **2015–2024**) for rotation analysis across the study stack.

**Data source (required):** `data/processed/cdl/cdl_stack_wide.parquet` + `cdl_stack_spatial_metadata.json` — **not** interim NetCDF. After `process_interim_to_parquet.py`, the wide file includes **`cdl_YYYY`** columns (e.g. **2008–2025** when the full stack is exported); this notebook uses only the years in `configs/task2_crop_rotation.yaml` → `cdl.year_range`.

**Outputs:** sanity tables, corn/soy fraction bar chart under **`artifacts/figures/task2/`** (from YAML `output.figures_dir`).

**Rigor (brief + Task 1 pattern):** Treat `configs/task2_crop_rotation.yaml` as the single source of truth for years, thresholds, and artifact directories — same idea as `notebooks/task1_ndvi_timeseries/03_ndvi_phenology_hsgp_bayesian.ipynb` (YAML + processed Parquet + JSON sidecars). Rubric items live in **`resources/CropSmart_NAFSI_Track1_Challenge_Brief.docx.pdf`**; see also **`context/TASK2_NAFSI_DATA_CONTRACT.md`** for paths and a reproducibility checklist.


In [1]:
import sys
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root.")
sys.path.insert(0, str(REPO_ROOT))

from src.io.cdl_parquet import (
    load_cdl_spatial_metadata,
    load_cdl_wide_years,
    year_range_inclusive,
)

with open(REPO_ROOT / "configs" / "task2_crop_rotation.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

YEARS = year_range_inclusive(cfg["cdl"]["year_range"])
corn = int(cfg["cdl"]["crop_classes"]["corn"])
soy = int(cfg["cdl"]["crop_classes"]["soybean"])
print("CDL years (inclusive):", YEARS[0], "–", YEARS[-1], "n=", len(YEARS))

meta = load_cdl_spatial_metadata(REPO_ROOT)
H, W = int(meta["height"]), int(meta["width"])
print("Grid:", H, "x", W, "| CRS:", meta.get("crs"), "| transform[0:6]:", (meta.get("transform") or [])[:6])

cdl_df = load_cdl_wide_years(REPO_ROOT, YEARS)
print("Parquet rows:", f"{len(cdl_df):,}", "(expect height*width)")
assert len(cdl_df) == H * W, "Row count mismatch — check processed CDL export."


CDL years (inclusive): 2015 – 2024 n= 10
Grid: 2960 x 4096 | CRS: EPSG:5070 | transform[0:6]: [556.6650844077274, 0.0, -789410.7827709025, 0.0, -556.6650844077274, 3078903.5769611606]


Parquet rows: 12,124,160 (expect height*width)


In [2]:
rows = []
for y in YEARS:
    col = f"cdl_{y}"
    s = cdl_df[col].to_numpy()
    rows.append(
        {
            "year": y,
            "frac_corn": float(np.mean(s == corn)),
            "frac_soy": float(np.mean(s == soy)),
            "frac_corn_or_soy": float(np.mean(np.isin(s, [corn, soy]))),
        }
    )

frac_tbl = pd.DataFrame(rows)
display(frac_tbl)

fig_dir = REPO_ROOT / cfg["output"]["figures_dir"]
fig_dir.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(9, 4))
x = frac_tbl["year"].astype(str)
ax.bar(x, frac_tbl["frac_corn"], label="Corn", color="#f1c40f", alpha=0.9)
ax.bar(x, frac_tbl["frac_soy"], bottom=frac_tbl["frac_corn"], label="Soybean", color="#27ae60", alpha=0.9)
ax.set_ylabel("Fraction of grid cells")
ax.set_xlabel("CDL year")
ax.set_title("Corn vs soybean area fraction by year (processed CDL stack)")
ax.legend()
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
out_png = fig_dir / f"task2__cornsoy_fractions_by_year__{date.today():%Y%m%d}.png"
fig.savefig(out_png, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", out_png.relative_to(REPO_ROOT))


,year,frac_corn,frac_soy,frac_corn_or_soy
0,2015,0.084057,0.073140,0.157196
1,2016,0.088875,0.074724,0.163599
2,2017,0.087085,0.082679,0.169765
3,2018,0.085298,0.083054,0.168352
4,2019,0.086355,0.071402,0.157757
5,2020,0.088386,0.078963,0.167350
6,2021,0.089940,0.082674,0.172613
7,2022,0.086713,0.081409,0.168122
8,2023,0.091729,0.077599,0.169328
9,2024,0.087636,0.079133,0.166769


Saved: artifacts\figures\task2\task2__cornsoy_fractions_by_year__20260411.png


C:\Users\sardo\AppData\Local\Temp\ipykernel_29256\942383836.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
